# Inspect Intan digital sync

This notebook loads an Intan `digitalin.dat` file recorded in the “One File Per Signal Type” format, extracts one digital input channel, and plots the recorded sync pulse train. Set the `MULTIMODAL_SYNC_EXAMPLE_SESSION` environment variable or edit `SESSION_BASEPATH` before running the notebook.

## Configure the input file

Use the `digitalin.dat` file from an Intan recording folder. The digital channel ID is zero-indexed and should match the physical DIN channel used for the Arduino sync pulse. If `INTAN_RECORDING_NAME` is `None`, the notebook uses the first subfolder inside `raw_intan`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


SESSION_BASEPATH = Path(os.environ.get("MULTIMODAL_SYNC_EXAMPLE_SESSION", "/path/to/example_session_01"))
INTAN_RECORDING_NAME = None
DIGITAL_CHANNEL_ID = 0
SAMPLE_RATE_HZ = 30_000

# Set to an integer to limit the number of points in full-session plots.
# Set to None to plot every sample.
FULL_PLOT_MAX_POINTS = 200_000

ZOOM_BEFORE_S = 0.05
ZOOM_AFTER_S = 0.25

## Load the digital input words

Intan stores board digital inputs 0-15 as bit fields in each `uint16` word of `digitalin.dat`. A single channel can be isolated with a bit mask.

In [ ]:
intan_root = SESSION_BASEPATH / "raw_intan"
if INTAN_RECORDING_NAME is None:
    intan_recording_dirs = sorted(p for p in intan_root.iterdir() if p.is_dir())
    if not intan_recording_dirs:
        raise FileNotFoundError(f"No Intan recording folders found in: {intan_root}")
    intan_recording_dir = intan_recording_dirs[0]
else:
    intan_recording_dir = intan_root / INTAN_RECORDING_NAME

DIGITALIN_PATH = intan_recording_dir / "digitalin.dat"
if not DIGITALIN_PATH.exists():
    raise FileNotFoundError(f"digitalin.dat not found: {DIGITALIN_PATH}")
if not 0 <= DIGITAL_CHANNEL_ID <= 15:
    raise ValueError("DIGITAL_CHANNEL_ID must be between 0 and 15")

digital_words = np.fromfile(DIGITALIN_PATH, dtype=np.uint16)
digital_sync = (digital_words & (1 << DIGITAL_CHANNEL_ID)) > 0
time_s = np.arange(digital_sync.size) / SAMPLE_RATE_HZ


def plot_indices(n_points, max_points=None):
    if max_points is None:
        return np.arange(n_points)
    stride = max(1, int(np.ceil(n_points / max_points)))
    return np.arange(0, n_points, stride)


print(f"file: {DIGITALIN_PATH}")
print(f"sample_rate: {SAMPLE_RATE_HZ} Hz")
print(f"samples: {digital_sync.size}")
print(f"duration: {digital_sync.size / SAMPLE_RATE_HZ:.3f} s")
print(f"digital channel: {DIGITAL_CHANNEL_ID}")
print(f"high samples: {digital_sync.sum()}")

## Plot the full recording

This view shows where the sync pulse train starts and stops within the full Intan recording.

In [ ]:
full_idx = plot_indices(digital_sync.size, FULL_PLOT_MAX_POINTS)

fig, ax = plt.subplots(figsize=(12, 3), constrained_layout=True)
ax.plot(time_s[full_idx], digital_sync[full_idx].astype(int), color="tab:red")
ax.set_title(f"Intan digital sync channel {DIGITAL_CHANNEL_ID}")
ax.set_xlabel("time (s)")
ax.set_ylabel("digital state")
ax.set_yticks([0, 1])
plt.show()

## Zoom in around the first rising edge

This plot zooms around the first detected rising edge so the square-wave pulse train can be inspected directly.

In [ ]:
rising_edges = np.flatnonzero(np.diff(digital_sync.astype(np.int8)) == 1) + 1
if rising_edges.size == 0:
    raise ValueError("No rising edges detected. Check wiring, RHX digital input settings, and the channel ID.")

first_edge = rising_edges[0]
zoom_start_s = max(0, time_s[first_edge] - ZOOM_BEFORE_S)
zoom_end_s = min(time_s[-1], time_s[first_edge] + ZOOM_AFTER_S)
zoom_mask = (time_s >= zoom_start_s) & (time_s <= zoom_end_s)

fig, ax = plt.subplots(figsize=(12, 3), constrained_layout=True)
ax.plot(time_s[zoom_mask], digital_sync[zoom_mask].astype(int), color="tab:red")
ax.set_title("Zoom around first Intan digital sync rising edge")
ax.set_xlabel("time (s)")
ax.set_ylabel("digital state")
ax.set_yticks([0, 1])
plt.show()

## Full-session figure with inset zoom

This final plot is useful for the protocol figure: it shows the full Intan digital sync recording, with an inset zoom around the first detected rising edge.

In [ ]:
full_idx = plot_indices(digital_sync.size, FULL_PLOT_MAX_POINTS)

fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
ax.plot(time_s[full_idx], digital_sync[full_idx].astype(int), color="tab:red")
ax.set_title(f"Intan digital sync channel {DIGITAL_CHANNEL_ID}")
ax.set_xlabel("time (s)")
ax.set_ylabel("digital state")
ax.set_yticks([0, 1])

inset_ax = ax.inset_axes([0.6, 0.35, 0.35, 0.5])
inset_ax.plot(time_s[zoom_mask], digital_sync[zoom_mask].astype(int), color="tab:red")
inset_ax.set_xlim(zoom_start_s, zoom_end_s)
inset_ax.set_ylim(-0.1, 1.1)
inset_ax.set_xticks([])
inset_ax.set_yticks([0, 1])
inset_ax.set_title("sync onset")
ax.indicate_inset_zoom(inset_ax, edgecolor="gray", alpha=1)

plt.show()